# Late Fusion — U-Net + ProfStyleLSTM (Prediction Level)

Fuses `unet_baseline.ipynb` (U-Net) with the ProfStyleLSTM from
`lstm_prof_style.ipynb` at the **prediction level**.

```
RGB     ---> U-Net  ---> unet_logits (B,3,128,128)  ─┐
                                                       alpha*unet + (1-alpha)*lstm
Segs    ---> LSTM(48 uni-dir) ---> last step (B,48) ─┘
                -> head(16,16,3) -> lstm_logits (B,3) tiled to (B,3,128,128)
```

`alpha` is a **learnable scalar** (initialised at 0.5).
`model.blend_weight()` shows the final branch trust ratio.


In [ ]:
PROJECT_ROOT = Path("/home/spant/Research Seminar/Project")
EXP_NAME     = "fusion_late_unet_profstyle_v1"


## 0. Setup

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

In [ ]:
import sys, subprocess
for pkg in ["segmentation_models_pytorch"]:
    try:
        __import__(pkg.replace("-","_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


In [ ]:
import json, random, re, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

print("torch:", torch.__version__,
      "cuda:", torch.cuda.is_available(),
      "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## 1. Config

In [ ]:
CSV_DIR    = PROJECT_ROOT / "IS2_Corrected_data"
IMG_DIR    = PROJECT_ROOT / "outputs"
MASK_DIR   = PROJECT_ROOT / "outputs_segmented"
PRIOR_UNET = PROJECT_ROOT / "runs" / "unet_imgonly_v1"
RUN_DIR    = PROJECT_ROOT / "runs" / EXP_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

SEED         = 42
NUM_CLASSES  = 3
PATCH        = 128
SEQ_LEN      = 5              # prof-style: 2 before + center + 2 after
NEARBY       = SEQ_LEN // 2   # 2
BATCH_SIZE   = 32
EPOCHS       = 30
LR           = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 7
NUM_WORKERS  = 4
USE_AMP      = True

# Prof-style LSTM arch (uni-directional)
LSTM_HIDDEN  = 48             # prof's LSTM units=48, UNI-directional -> output dim=48
LSTM_OUT     = LSTM_HIDDEN    # no *2 since not bidirectional
LSTM_DROPOUT = 0.4
HEAD_HIDDEN  = 16             # prof: Dense(16, elu)
HEAD_DROPOUT = 0.4
FUSION_CH    = 16             # intermediate channels before classification head
GRAD_CLIP    = 1.0

ALPHA_FL = [0.05, 0.45, 0.60]  # focal loss class weights
GAMMA_FL = 2.0

CLASS_NAMES  = ["ice", "thin_ice", "water"]
CLASS_COLORS = {0: (255,0,0), 1: (0,0,255), 2: (0,255,0)}
IM_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IM_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

FEATURES = ["h_cor_mean", "h_diff", "rel_height_min_elev", "height_sd",
            "pcnth_mean", "pcnt_mean", "bcnt_mean", "brate_mean"]
N_FEATS  = len(FEATURES)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("Run dir:", RUN_DIR)
print(f"SEQ_LEN={SEQ_LEN}  LSTM uni-dir hidden={LSTM_HIDDEN}  LSTM_OUT={LSTM_OUT}")
print(f"FUSION_CH={FUSION_CH}  GRAD_CLIP={GRAD_CLIP}")


## 2. Manifest

Re-use the manifest from `unet_imgonly_v1` (same 128x128 patches).
Each row links an image patch to its tile, beam, and center row-index
in the `ATL03_*_done.csv` segment file.

In [ ]:
mp = PRIOR_UNET / "manifest.csv"
if mp.exists():
    manifest = pd.read_csv(mp)
    print(f"Loaded manifest: {len(manifest):,} rows")
else:
    pat = re.compile(r"^row(\d+)_(\d{8}T\d{6})_(T\d+[A-Z]+)_(gt[12]r)\.png$")
    rows = []
    for p in sorted(IMG_DIR.iterdir()):
        m = pat.match(p.name)
        if not m: continue
        rows.append({"filename": p.name, "row_idx": int(m.group(1)),
                     "date": m.group(2), "tile": m.group(3), "beam": m.group(4),
                     "image_path": str(p), "mask_path": str(MASK_DIR / p.name)})
    manifest = pd.DataFrame(rows)
    manifest.to_csv(RUN_DIR / "manifest.csv", index=False)
    print(f"Built manifest: {len(manifest):,} rows")


## 3. Load 10 m segment CSVs and engineer features

Same 8 features as `lstm_prof_style.ipynb`:
`h_cor_mean`, `h_diff` (= h_cor_mean - h_cor_med),
`rel_height_min_elev` (= h_cor_mean - min per track),
`height_sd`, `pcnth_mean`, `pcnt_mean`, `bcnt_mean`, `brate_mean`.

In [ ]:
csv_files = sorted(CSV_DIR.glob("ATL03_*_done.csv"))
csv_meta  = []
for p in csv_files:
    parts = p.stem.split("_")
    csv_meta.append({"csv_path": str(p), "tile": parts[3], "beam": parts[4]})
csv_meta = pd.DataFrame(csv_meta)
csv_meta["csv_id"] = csv_meta.index

manifest = manifest.merge(csv_meta[["tile","beam","csv_id"]], on=["tile","beam"], how="left")
manifest["csv_id"] = manifest["csv_id"].astype(int)
print(f"Linked {manifest['csv_id'].nunique()} CSVs to manifest")

needed_cols = ["h_cor_mean","h_cor_med","x_atc","height_sd",
               "pcnth_mean","pcnt_mean","bcnt_mean","brate_mean"]
seg_feats = {}   # csv_id -> (N_seg, N_FEATS) float32

for _, row in csv_meta.iterrows():
    df = pd.read_csv(row["csv_path"])
    miss = [c for c in needed_cols if c not in df.columns]
    if miss:
        print(f"  SKIP {row['csv_path']}: missing {miss}"); continue
    df = df.sort_values("x_atc").reset_index(drop=True)
    df["h_diff"]              = df["h_cor_mean"] - df["h_cor_med"]
    df["rel_height_min_elev"] = df["h_cor_mean"] - df["h_cor_mean"].min()
    seg_feats[int(row["csv_id"])] = df[FEATURES].to_numpy(dtype=np.float32)
    print(f"  csv_{int(row['csv_id'])}: {len(df):,} segs")


## 4. Standardize features (train tiles only)

In [ ]:
tiles_train = {"T02CNA","T02CNC"}
tiles_test  = {"T03CWT"}

train_df = manifest[manifest["tile"].isin(tiles_train)]
rng = np.random.RandomState(SEED)
val_idx = rng.choice(len(train_df), size=int(0.10*len(train_df)), replace=False)
vm = np.zeros(len(train_df), dtype=bool); vm[val_idx] = True
val_df   = train_df.iloc[vm].reset_index(drop=True)
train_df = train_df.iloc[~vm].reset_index(drop=True)
test_df  = manifest[manifest["tile"].isin(tiles_test)].reset_index(drop=True)
print(f"Train {len(train_df):,}  Val {len(val_df):,}  Test {len(test_df):,}")

# Compute mean/std from all train-tile feature vectors (no validity filter needed)
train_feats = []
for cid, feats in seg_feats.items():
    tile = csv_meta.loc[csv_meta["csv_id"]==cid, "tile"].values
    if len(tile) and tile[0] in tiles_train:
        train_feats.append(feats)

if train_feats:
    tf  = np.concatenate(train_feats, axis=0)
    seg_means = tf.mean(0).astype(np.float32)
    seg_stds  = (tf.std(0) + 1e-6).astype(np.float32)
else:
    seg_means = np.zeros(N_FEATS, dtype=np.float32)
    seg_stds  = np.ones(N_FEATS,  dtype=np.float32)

for cid in seg_feats:
    seg_feats[cid] = (seg_feats[cid] - seg_means[None,:]) / seg_stds[None,:]

np.save(RUN_DIR / "seg_means.npy", seg_means)
np.save(RUN_DIR / "seg_stds.npy",  seg_stds)
print("means:", seg_means.round(3)); print("stds: ", seg_stds.round(3))


## 5. Dataset

Returns `(image, seg_window, pixel_mask)`:
- `image`      (3, 128, 128) normalized Sentinel-2 patch
- `seg_window` (SEQ_LEN=5, 8) standardized segment features
- `pixel_mask` (128, 128) int64 per-pixel class labels

No `valid_mask` needed — `ProfStyleLSTM` processes all steps equally
(last-step pooling, no sparsity handling).

In [ ]:
def mask_rgb_to_int(m):
    out = np.full(m.shape[:2], 255, dtype=np.uint8)
    out[(m==[255,0,0]).all(-1)] = 0
    out[(m==[0,0,255]).all(-1)] = 1
    out[(m==[0,255,0]).all(-1)] = 2
    return out


def aug_flip_rot(img, mask):
    if random.random() < 0.5:
        img = img[:,::-1,:]; mask = mask[:,::-1]
    if random.random() < 0.5:
        img = img[::-1,:,:]; mask = mask[::-1,:]
    k = random.randint(0,3)
    if k:
        img  = np.rot90(img,  k, axes=(0,1))
        mask = np.rot90(mask, k, axes=(0,1))
    return np.ascontiguousarray(img), np.ascontiguousarray(mask)


class UNetProfDataset(Dataset):
    def __init__(self, df, seg_feats, augment=False):
        self.df        = df.reset_index(drop=True)
        self.seg_feats = seg_feats
        self.augment   = augment

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r   = self.df.iloc[i]
        img = np.array(Image.open(r["image_path"]).convert("RGB"))
        msk = mask_rgb_to_int(np.array(Image.open(r["mask_path"]).convert("RGB")))
        if self.augment:
            img, msk = aug_flip_rot(img, msk)
        img = ((img.astype(np.float32)/255.0 - IM_MEAN) / IM_STD)
        img = np.transpose(img, (2,0,1))

        feats  = self.seg_feats[int(r["csv_id"])]
        n_seg  = len(feats)
        ctr    = int(r["row_idx"])
        win    = np.zeros((SEQ_LEN, N_FEATS), dtype=np.float32)
        for k in range(SEQ_LEN):
            src = ctr - NEARBY + k
            if 0 <= src < n_seg:
                win[k] = feats[src]

        return (torch.from_numpy(img),
                torch.from_numpy(win),
                torch.from_numpy(msk).long())


## 6. DataLoaders

In [ ]:
train_ds = UNetProfDataset(train_df, seg_feats, augment=True)
val_ds   = UNetProfDataset(val_df,   seg_feats, augment=False)
test_ds  = UNetProfDataset(test_df,  seg_feats, augment=False)

kw = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS>0)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True, **kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **kw)
print(f"train {len(train_loader)} steps  val {len(val_loader)}  test {len(test_loader)}")

_x, _w, _m = train_ds[0]
print(f"img {tuple(_x.shape)}  seg_win {tuple(_w.shape)}  mask {tuple(_m.shape)}")


## 7. Device

In [ ]:
import time

## 8. Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class LateFusionUNetProf(nn.Module):
    # Prediction-level fusion: U-Net per-pixel logits + ProfStyleLSTM
    # per-patch logits (tiled to spatial size), blended by learnable alpha.
    #
    # ProfStyleLSTM branch: uni-directional LSTM(48), last-step -> head.
    # Head mirrors the prof's Dense(16,elu) x2 -> Dense(3).
    def __init__(self, n_features=N_FEATS):
        super().__init__()
        # Image branch: full U-Net producing NUM_CLASSES logits
        self.unet = smp.Unet("resnet18", encoder_weights="imagenet",
                              in_channels=3, classes=NUM_CLASSES)
        # CSV branch: uni-directional LSTM(48) + classification head
        self.lstm      = nn.LSTM(n_features, LSTM_HIDDEN, num_layers=1,
                                 batch_first=True, bidirectional=False)
        self.lstm_drop = nn.Dropout(LSTM_DROPOUT)
        self.lstm_head = nn.Sequential(
            nn.Linear(LSTM_OUT, HEAD_HIDDEN), nn.ELU(inplace=True),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(HEAD_HIDDEN, HEAD_HIDDEN), nn.ELU(inplace=True),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(HEAD_HIDDEN, NUM_CLASSES),
        )
        # Learnable blend weight (sigmoid -> (0,1))
        self.alpha = nn.Parameter(torch.tensor(0.0))

    def forward(self, img, seg_win):
        # U-Net per-pixel logits
        unet_logits = self.unet(img)                   # (B, 3, H, W)

        # LSTM per-patch logits
        out, _      = self.lstm(seg_win)               # (B, SEQ_LEN, 48)
        feat        = self.lstm_drop(out[:, -1, :])    # last step: (B, 48)
        lstm_logits = self.lstm_head(feat)             # (B, 3)
        lstm_logits = lstm_logits[:,:,None,None].expand_as(unet_logits)

        a = torch.sigmoid(self.alpha)
        return a * unet_logits + (1.-a) * lstm_logits

    def blend_weight(self):
        a = torch.sigmoid(self.alpha).item()
        return {"unet_weight": round(a,4), "lstm_weight": round(1-a,4)}


model  = LateFusionUNetProf()
n_p    = sum(p.numel() for p in model.parameters())
print(f"LateFusionUNetProf  total={n_p:,}")


## 9. Focal Loss + AdamW + CosineAnnealing

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0, ignore=255):
        super().__init__()
        self.register_buffer("alpha", torch.tensor(alpha, dtype=torch.float32))
        self.gamma = float(gamma); self.ignore = ignore
    def forward(self, logits, target):
        valid  = target != self.ignore
        t_safe = target.clamp_min(0)
        lp  = F.log_softmax(logits, dim=1)
        lpt = lp.gather(1, t_safe.unsqueeze(1)).squeeze(1)
        pt  = lpt.exp().clamp(1e-8, 1-1e-8)
        at  = self.alpha.to(logits.device)[t_safe]
        loss = -at * (1-pt).pow(self.gamma) * lpt
        return (loss * valid.float()).sum() / valid.float().sum().clamp(min=1.)


device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = model.to(device)
criterion = FocalLoss(alpha=ALPHA_FL, gamma=GAMMA_FL).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler("cuda", enabled=USE_AMP)
print("criterion / optimizer / scheduler ready")


## 10. Training loop

In [ ]:
def cm_step(cm, logits, targets):
    pred = logits.argmax(1).detach().cpu().numpy().ravel()
    t    = targets.detach().cpu().numpy().ravel()
    keep = t != 255
    pred, t = pred[keep], t[keep]
    cm += np.bincount(NUM_CLASSES*t+pred,
                      minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)


def miou_from_cm(cm):
    iou = []
    for c in range(NUM_CLASSES):
        tp=cm[c,c]; fp=cm[:,c].sum()-tp; fn=cm[c,:].sum()-tp
        d=tp+fp+fn; iou.append(float(tp/d) if d else 0.)
    iou = np.array(iou, dtype=np.float64)
    return float(iou.mean()), iou, float(np.diag(cm).sum()/max(cm.sum(),1))


def prf1_from_cm(cm):
    # Per-class precision, recall, F1 from confusion matrix.
    prec=np.zeros(NUM_CLASSES); rec=np.zeros(NUM_CLASSES); f1=np.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        tp=cm[c,c]; fp=cm[:,c].sum()-tp; fn=cm[c,:].sum()-tp
        prec[c] = tp/(tp+fp) if (tp+fp) else 0.
        rec[c]  = tp/(tp+fn) if (tp+fn) else 0.
        f1[c]   = 2*prec[c]*rec[c]/(prec[c]+rec[c]) if (prec[c]+rec[c]) else 0.
    return prec, rec, f1


best_val      = -1.; patience_left = PATIENCE; log = []

for epoch in range(1, EPOCHS+1):
    t0 = time.perf_counter()
    model.train()
    tr_loss=0.; n_seen=0; tr_cm=np.zeros((NUM_CLASSES,NUM_CLASSES),dtype=np.int64)
    for img, win, msk in train_loader:
        img  = img.to(device,  non_blocking=True)
        win  = win.to(device,  non_blocking=True)
        msk  = msk.to(device,  non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits = model(img, win)
            loss   = criterion(logits, msk)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer); scaler.update()
        tr_loss += loss.item()*img.size(0); n_seen += img.size(0)
        cm_step(tr_cm, logits, msk)
    tr_loss /= max(n_seen,1); tr_miou,_,tr_acc = miou_from_cm(tr_cm)
    tr_prec, tr_rec, tr_f1 = prf1_from_cm(tr_cm)

    model.eval(); va_loss=0.; n_seen=0; va_cm=np.zeros((NUM_CLASSES,NUM_CLASSES),dtype=np.int64)
    with torch.no_grad():
        for img, win, msk in val_loader:
            img=img.to(device,non_blocking=True); win=win.to(device,non_blocking=True)
            msk=msk.to(device,non_blocking=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                logits = model(img, win)
                va_loss += criterion(logits,msk).item()*img.size(0)
            n_seen += img.size(0); cm_step(va_cm, logits, msk)
    va_loss /= max(n_seen,1); va_miou, va_iou, va_acc = miou_from_cm(va_cm)
    va_prec, va_rec, va_f1 = prf1_from_cm(va_cm)
    scheduler.step()
    lr = optimizer.param_groups[0]["lr"]

    log.append({"epoch":epoch,
                "train_loss":tr_loss,"train_miou":tr_miou,"train_acc":tr_acc,
                "train_macro_f1":float(tr_f1.mean()),
                "val_loss":va_loss,"val_miou":va_miou,"val_acc":va_acc,
                "val_ice":va_iou[0],"val_thin":va_iou[1],"val_water":va_iou[2],
                "val_prec_ice":va_prec[0],"val_prec_thin":va_prec[1],"val_prec_water":va_prec[2],
                "val_rec_ice":va_rec[0],"val_rec_thin":va_rec[1],"val_rec_water":va_rec[2],
                "val_f1_ice":va_f1[0],"val_f1_thin":va_f1[1],"val_f1_water":va_f1[2],
                "val_macro_f1":float(va_f1.mean()),"lr":lr})
    print(f"ep {epoch:03d}  tr {tr_loss:.4f}/mIoU={tr_miou:.4f}/F1={tr_f1.mean():.4f}  "
          f"va {va_loss:.4f}/mIoU={va_miou:.4f}/F1={va_f1.mean():.4f}  "
          f"acc={va_acc:.4f}  [{va_iou.round(3).tolist()}]  "
          f"lr={lr:.2e}  ({time.perf_counter()-t0:.0f}s)")

    if va_miou > best_val + 1e-4:
        best_val = va_miou; patience_left = PATIENCE
        torch.save({"epoch":epoch,"model_state":model.state_dict(),
                    "val_metrics":{"miou":va_miou,"per_iou":va_iou.tolist(),
                                   "pix_acc":va_acc,
                                   "macro_f1":float(va_f1.mean()),
                                   "per_f1":va_f1.tolist(),
                                   "per_prec":va_prec.tolist(),
                                   "per_rec":va_rec.tolist()}}, RUN_DIR/"best.pt")
    else:
        patience_left -= 1
        if patience_left <= 0:
            print(f"Early stop epoch {epoch}  best val mIoU={best_val:.4f}"); break

pd.DataFrame(log).to_csv(RUN_DIR/"metrics.csv", index=False)
print(f"\nBest val mIoU: {best_val:.4f}")


## 11. Test evaluation (best.pt)

In [ ]:
ck = torch.load(RUN_DIR/"best.pt", map_location=device, weights_only=False)
model.load_state_dict(ck["model_state"]); model.eval()
vm = ck["val_metrics"]
print(f"Loaded epoch {ck['epoch']}  "
      f"val mIoU={vm['miou']:.4f}  macro-F1={vm.get('macro_f1',float('nan')):.4f}  "
      f"pix_acc={vm['pix_acc']:.4f}")

te_cm=np.zeros((NUM_CLASSES,NUM_CLASSES),dtype=np.int64); te_loss=0.; n_seen=0
with torch.no_grad():
    for img, win, msk in test_loader:
        img=img.to(device,non_blocking=True); win=win.to(device,non_blocking=True)
        msk=msk.to(device,non_blocking=True)
        with torch.amp.autocast("cuda", enabled=USE_AMP):
            logits = model(img, win)
            te_loss += criterion(logits,msk).item()*img.size(0)
        n_seen += img.size(0); cm_step(te_cm, logits, msk)
te_loss /= max(n_seen,1)
te_miou, te_iou, te_acc = miou_from_cm(te_cm)
prec, rec, f1 = prf1_from_cm(te_cm)

header = f"{'class':12s}  {'IoU':>7}  {'Prec':>7}  {'Rec':>7}  {'F1':>7}"
print(f"\nTEST  mIoU={te_miou:.4f}  pix_acc={te_acc:.4f}  macro-F1={f1.mean():.4f}  loss={te_loss:.4f}")
print(header); print("-"*len(header))
for name,iou,p,r,f in zip(CLASS_NAMES,te_iou,prec,rec,f1):
    print(f"{name:12s}  {iou:7.4f}  {p:7.4f}  {r:7.4f}  {f:7.4f}")
print(f"{'macro avg':12s}  {te_miou:7.4f}  {prec.mean():7.4f}  {rec.mean():7.4f}  {f1.mean():7.4f}")

with open(RUN_DIR/"test_metrics.json","w") as fout:
    json.dump({"miou":te_miou,"per_iou":te_iou.tolist(),"pix_acc":te_acc,"loss":te_loss,
               "precision":prec.tolist(),"recall":rec.tolist(),"f1":f1.tolist(),
               "macro_f1":float(f1.mean()),"macro_prec":float(prec.mean()),
               "macro_rec":float(rec.mean())}, fout, indent=2)


## 11b. Metrics bar chart — per-class IoU, Precision, Recall, F1

In [ ]:
metrics_arr  = np.array([te_iou, prec, rec, f1])   # (4, 3)
metric_names = ["IoU", "Precision", "Recall", "F1"]
x = np.arange(NUM_CLASSES); w = 0.2

fig, ax = plt.subplots(figsize=(9, 4.5))
for i, (m_vals, m_name) in enumerate(zip(metrics_arr, metric_names)):
    ax.bar(x + i*w, m_vals, width=w, label=m_name, alpha=0.85)

ax.set_xticks(x + 1.5*w); ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
ax.set_title(f"Test metrics per class — {EXP_NAME}", fontsize=12)
ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)

for i, (v, name) in enumerate(zip([te_miou, prec.mean(), rec.mean(), f1.mean()], metric_names)):
    ax.axhline(v, color=f"C{i}", linestyle=":", linewidth=1, alpha=0.6)

plt.tight_layout()
plt.savefig(RUN_DIR/"metrics_bar.png", dpi=160, bbox_inches="tight"); plt.show()


## 12. Confusion matrix (prof-style percentage view)

In [ ]:
def plot_cm(cm, title, path):
    pct = cm.astype(np.float64)
    rs  = pct.sum(1,keepdims=True)
    pct = np.where(rs>0, pct/np.maximum(rs,1)*100, 0.)
    fig,ax = plt.subplots(figsize=(7.2,5.8))
    im = ax.imshow(pct, cmap="Blues", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels([f"Pred {c}" for c in CLASS_NAMES], fontsize=11)
    ax.set_yticklabels(CLASS_NAMES, fontsize=11)
    ax.set_xlabel("Predicted",fontsize=12); ax.set_ylabel("Actual",fontsize=12)
    ax.set_title(title, fontsize=13)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            v=pct[i,j]; ax.text(j,i,f"{v:.2f}",ha="center",va="center",
                                color="white" if v>55 else "black",fontsize=13)
    fig.colorbar(im,ax=ax,fraction=0.046,pad=0.04); plt.tight_layout()
    plt.savefig(path,dpi=180,bbox_inches="tight"); plt.show()

plot_cm(te_cm, f"Confusion Matrix (Percentages) — {EXP_NAME}", RUN_DIR/"confmat.png")


## 13. Loss, mIoU, F1 + accuracy curves

In [ ]:
hist = pd.read_csv(RUN_DIR/"metrics.csv")
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.ravel()

axes[0].plot(hist.epoch, hist.train_loss, label="train")
axes[0].plot(hist.epoch, hist.val_loss,   label="val")
axes[0].set_title("Focal loss"); axes[0].set_xlabel("epoch")
axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(hist.epoch, hist.train_miou, label="train mIoU")
axes[1].plot(hist.epoch, hist.val_miou,   label="val mIoU")
axes[1].plot(hist.epoch, hist.val_thin, "--", label="thin_ice IoU", alpha=.7)
axes[1].plot(hist.epoch, hist.val_acc,  ":",  label="val pix_acc",  alpha=.7)
axes[1].set_title("mIoU + pixel accuracy"); axes[1].set_xlabel("epoch")
axes[1].legend(); axes[1].grid(alpha=.3)

axes[2].plot(hist.epoch, hist.train_macro_f1, label="train macro-F1", linewidth=2)
axes[2].plot(hist.epoch, hist.val_macro_f1,   label="val macro-F1",   linewidth=2)
axes[2].plot(hist.epoch, hist.val_f1_ice,   "--", label="F1 ice",      alpha=.75)
axes[2].plot(hist.epoch, hist.val_f1_thin,  "--", label="F1 thin_ice", alpha=.75)
axes[2].plot(hist.epoch, hist.val_f1_water, "--", label="F1 water",    alpha=.75)
axes[2].set_title("F1 scores (val per-class + macro)"); axes[2].set_xlabel("epoch")
axes[2].legend(fontsize=8); axes[2].grid(alpha=.3)

for col, name, ls in zip(["ice","thin","water"], CLASS_NAMES, ["-","--",":"]):
    axes[3].plot(hist.epoch, hist[f"val_prec_{col}"],
                 ls, label=f"prec {name}", alpha=.8)
    axes[3].plot(hist.epoch, hist[f"val_rec_{col}"],
                 ls, label=f"rec {name}",  alpha=.5)
axes[3].set_title("Val precision & recall per class"); axes[3].set_xlabel("epoch")
axes[3].legend(fontsize=7); axes[3].grid(alpha=.3)

plt.suptitle(EXP_NAME, fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(RUN_DIR/"loss_curve.png",dpi=160,bbox_inches="tight"); plt.show()


## 14. Sample predictions

In [ ]:
def int_to_rgb(m):
    out = np.zeros((*m.shape,3), dtype=np.uint8)
    for c, col in CLASS_COLORS.items(): out[m==c] = col
    return out


model.eval()
sample = test_df.sample(n=6, random_state=SEED).reset_index(drop=True)
fig, axes = plt.subplots(6, 3, figsize=(7.5,14))
with torch.no_grad():
    for i, r in sample.iterrows():
        rgb = np.array(Image.open(r["image_path"]).convert("RGB"))
        gt  = mask_rgb_to_int(np.array(Image.open(r["mask_path"]).convert("RGB")))

        img_t = torch.from_numpy(
            np.transpose((rgb/255.-IM_MEAN)/IM_STD, (2,0,1)
        ).astype(np.float32))[None].to(device)

        feats = seg_feats[int(r["csv_id"])]; n_seg = len(feats); ctr = int(r["row_idx"])
        win = np.zeros((SEQ_LEN,N_FEATS),dtype=np.float32)
        for k in range(SEQ_LEN):
            src = ctr-NEARBY+k
            if 0<=src<n_seg: win[k]=feats[src]
        win_t = torch.from_numpy(win)[None].to(device)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            pred = model(img_t, win_t).argmax(1)[0].cpu().numpy()

        axes[i,0].imshow(rgb);            axes[i,0].set_title("input"    if i==0 else "")
        axes[i,1].imshow(int_to_rgb(gt)); axes[i,1].set_title("GT"       if i==0 else "")
        axes[i,2].imshow(int_to_rgb(pred));axes[i,2].set_title(EXP_NAME  if i==0 else "")
        for ax in axes[i]: ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig(RUN_DIR/"sample_preds.png",dpi=150); plt.show()


In [ ]:
ck2 = torch.load(RUN_DIR/"best.pt", map_location=device, weights_only=False)
model.load_state_dict(ck2["model_state"])
print("Learned blend weights:", model.blend_weight())
